# TSP Baseline -- LKH-3 (Lin-Kernighan Heuristic)

LKH-3 is the state-of-the-art heuristic solver for TSP and related problems. It produces near-optimal solutions (typically within 0.5% of optimal) and scales to thousands of nodes. It serves as the primary baseline for large TSP instances where exact solvers like Concorde are too slow.

In [ ]:
%%bash
# Download and compile LKH-3
if [ ! -f /usr/local/bin/LKH ]; then
    cd /tmp
    wget -q http://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.13.tgz
    tar xzf LKH-3.0.13.tgz
    cd LKH-3.0.13
    make -j$(nproc) 2>&1 | tail -1
    cp LKH /usr/local/bin/
    echo "LKH-3 installed successfully"
else
    echo "LKH-3 already installed"
fi
LKH --version 2>&1 | head -1 || echo "LKH-3 ready"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import time
import subprocess
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Configuration
TSP_SIZES = list(range(10, 201, 10))
NUM_EVAL_INSTANCES = 128
LKH_RUNS = 1  # Number of LKH runs per instance (more runs = better solution but slower)
LKH_MAX_TRIALS = 1000  # Max improvement trials per run
TIME_LIMIT_SEC = 60  # Time limit per instance
DISTANCE_SCALE = 100_000
DRIVE_DATA_DIR = '/content/drive/MyDrive/TSP_Data'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/TSP_Data/baseline_results'
SEED = 42
LKH_BIN = '/usr/local/bin/LKH'
TEMP_DIR = '/tmp/lkh_work'  # temp dir for problem files

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

print(f"Configuration set. Results will be saved to: {DRIVE_RESULTS_DIR}")

In [ ]:
def compute_tour_length_euclidean(coords, tour):
    """Computes the Euclidean length of a tour."""
    if not tour:
        return float('inf')
    tour_coords = coords[tour]
    # Add the return to start
    tour_coords = np.vstack((tour_coords, tour_coords[0]))
    diffs = np.diff(tour_coords, axis=0)
    distances = np.linalg.norm(diffs, axis=1)
    return float(np.sum(distances))

def write_tsplib_file(coords, filepath, scale=DISTANCE_SCALE):
    """Writes coordinates to a TSPLIB format file."""
    n = len(coords)
    with open(filepath, 'w') as f:
        f.write(f"NAME: tsp_instance\n")
        f.write(f"TYPE: TSP\n")
        f.write(f"DIMENSION: {n}\n")
        f.write(f"EDGE_WEIGHT_TYPE: EUC_2D\n")
        f.write(f"NODE_COORD_SECTION\n")
        for i in range(n):
            x = int(coords[i, 0] * scale)
            y = int(coords[i, 1] * scale)
            f.write(f"{i + 1} {x} {y}\n")
        f.write("EOF\n")

def write_par_file(tsp_path, tour_path, par_path, runs=LKH_RUNS, max_trials=LKH_MAX_TRIALS, time_limit=TIME_LIMIT_SEC, seed=None):
    """Writes LKH-3 parameter file."""
    with open(par_path, 'w') as f:
        f.write(f"PROBLEM_FILE = {tsp_path}\n")
        f.write(f"OUTPUT_TOUR_FILE = {tour_path}\n")
        f.write(f"RUNS = {runs}\n")
        f.write(f"MAX_TRIALS = {max_trials}\n")
        f.write(f"TIME_LIMIT = {time_limit}\n")
        if seed is not None:
            f.write(f"SEED = {seed}\n")
        f.write(f"TRACE_LEVEL = 0\n")

def parse_lkh_tour(tour_path):
    """Parses the generated tour file from LKH-3."""
    tour = []
    reading = False
    with open(tour_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line == 'TOUR_SECTION':
                reading = True
                continue
            if line == '-1' or line == 'EOF':
                reading = False
                continue
            if reading:
                tour.append(int(line) - 1)  # Convert 1-indexed to 0-indexed
    return tour

def solve_tsp_lkh(coords, time_limit_sec=TIME_LIMIT_SEC, instance_id=0):
    """Solves a TSP instance using LKH-3."""
    n = len(coords)
    work_dir = os.path.join(TEMP_DIR, f'inst_{instance_id}')
    os.makedirs(work_dir, exist_ok=True)
    
    tsp_path = os.path.join(work_dir, 'problem.tsp')
    par_path = os.path.join(work_dir, 'params.par')
    tour_path = os.path.join(work_dir, 'solution.tour')
    
    write_tsplib_file(coords, tsp_path)
    write_par_file(tsp_path, tour_path, par_path,
                   time_limit=time_limit_sec, seed=SEED)
    
    t0 = time.time()
    try:
        result = subprocess.run(
            [LKH_BIN, par_path],
            capture_output=True, text=True,
            timeout=time_limit_sec + 30  # extra buffer
        )
        wall_time = time.time() - t0
        
        if os.path.exists(tour_path):
            tour = parse_lkh_tour(tour_path)
            tour_length = compute_tour_length_euclidean(coords, tour)
            status = 'SUCCESS'
        else:
            tour = []
            tour_length = float('inf')
            status = 'NO_SOLUTION'
    except subprocess.TimeoutExpired:
        wall_time = time.time() - t0
        if os.path.exists(tour_path):
            tour = parse_lkh_tour(tour_path)
            tour_length = compute_tour_length_euclidean(coords, tour)
            status = 'TIMEOUT_WITH_SOLUTION'
        else:
            tour = []
            tour_length = float('inf')
            status = 'TIMEOUT'
    except Exception as e:
        wall_time = time.time() - t0
        tour = []
        tour_length = float('inf')
        status = f'ERROR: {str(e)[:50]}'
    finally:
        # Cleanup temp files
        shutil.rmtree(work_dir, ignore_errors=True)
    
    return {
        'tour': tour,
        'tour_length': tour_length,
        'status': status,
        'wall_time': wall_time,
    }

In [ ]:
# Quick Smoke Test
print("Running Smoke Test on TSP-10...")
np.random.seed(SEED)
sample_coords = np.random.rand(10, 2)
res = solve_tsp_lkh(sample_coords, time_limit_sec=5, instance_id=9999)

print(f"Status: {res['status']}")
print(f"Tour Length: {res['tour_length']:.6f}")
print(f"Wall Time: {res['wall_time']:.4f}s")
print(f"Tour: {res['tour']}")
if res['status'] == 'SUCCESS':
    print("[OK] Smoke test passed.")
else:
    print("[WARNING] Smoke test failed.")

In [ ]:
all_results = []
lengths_dict = {}

for n in TSP_SIZES:
    print(f"\n{'='*40}")
    print(f"Evaluating TSP-{n}")
    print(f"{'='*40}")
    
    val_path = os.path.join(DRIVE_DATA_DIR, f'tsp_{n}_val.npy')
    if not os.path.exists(val_path):
        print(f"[WARNING] Validation file missing: {val_path}. Skipping.")
        continue
    
    dataset = np.load(val_path)
    
    num_to_eval = min(NUM_EVAL_INSTANCES, len(dataset))
    dataset = dataset[:num_to_eval]
    
    lengths = []
    times = []
    statuses = []
    
    for idx, coords in enumerate(tqdm(dataset, desc=f"TSP-{n} LKH-3")):
        res = solve_tsp_lkh(coords, time_limit_sec=TIME_LIMIT_SEC, instance_id=idx)
        
        lengths.append(res['tour_length'])
        times.append(res['wall_time'])
        statuses.append(res['status'])
        
        all_results.append({
            'n': n,
            'instance_id': idx,
            'status': res['status'],
            'tour_length': res['tour_length'],
            'wall_time': res['wall_time']
        })
    
    valid_lengths = [l for l, s in zip(lengths, statuses) if 'SUCCESS' in s or 'TIMEOUT_WITH_SOLUTION' in s]
    lengths_dict[n] = np.array(valid_lengths) if valid_lengths else np.array([])
    
    if valid_lengths:
        mean_len = np.mean(valid_lengths)
        mean_time = np.mean(times)
        success_rate = len(valid_lengths) / num_to_eval * 100
        print(f"Results for TSP-{n}:")
        print(f"  Mean Length: {mean_len:.6f}")
        print(f"  Mean Time:   {mean_time:.4f}s")
        print(f"  Success:     {success_rate:.1f}%")

In [ ]:
df_results = pd.DataFrame(all_results)
if len(df_results) > 0:
    summary = df_results.groupby('n').agg(
        mean_length=('tour_length', lambda x: np.mean(x[x < float('inf')])),
        mean_time=('wall_time', 'mean'),
        success_rate=('status', lambda x: (x.str.contains('SUCCESS') | x.str.contains('TIMEOUT_WITH_SOLUTION')).mean() * 100),
        num_instances=('n', 'count')
    ).reset_index()
    
    print("\nLKH-3 Summary:")
    print("-" * 60)
    print(summary.to_string(index=False))
    print("-" * 60)
else:
    print("[WARNING] No results to summarize.")

In [ ]:
if len(df_results) > 0:
    fig, axs = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Tour Length vs N
    axs[0].plot(summary['n'], summary['mean_length'], marker='o')
    axs[0].set_title('Mean Tour Length vs Problem Size (N)')
    axs[0].set_xlabel('N')
    axs[0].set_ylabel('Mean Tour Length')
    axs[0].grid(True)
    
    # 2. Wall Time vs N
    axs[1].plot(summary['n'], summary['mean_time'], marker='o', color='orange')
    axs[1].set_title('Mean Wall Time vs Problem Size (N)')
    axs[1].set_xlabel('N')
    axs[1].set_ylabel('Mean Wall Time (s)')
    axs[1].grid(True)
    
    # 3. Success Rate vs N
    axs[2].plot(summary['n'], summary['success_rate'], marker='o', color='green')
    axs[2].set_title('Success Rate vs Problem Size (N)')
    axs[2].set_xlabel('N')
    axs[2].set_ylabel('Success Rate (%)')
    axs[2].set_ylim(-5, 105)
    axs[2].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if lengths_dict:
    plt.figure(figsize=(12, 6))
    data_to_plot = [lengths_dict[n] for n in sorted(lengths_dict.keys()) if len(lengths_dict[n]) > 0]
    labels = [f"{n}" for n in sorted(lengths_dict.keys()) if len(lengths_dict[n]) > 0]
    
    if data_to_plot:
        plt.boxplot(data_to_plot, labels=labels)
        plt.title('Tour Length Distribution per Problem Size (LKH-3)')
        plt.xlabel('Problem Size (N)')
        plt.ylabel('Tour Length')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
if len(df_results) > 0:
    csv_path = os.path.join(DRIVE_RESULTS_DIR, 'lkh3_per_instance.csv')
    df_results.to_csv(csv_path, index=False)
    
    summary_path = os.path.join(DRIVE_RESULTS_DIR, 'lkh3_summary.csv')
    summary.to_csv(summary_path, index=False)
    
    for n, lengths in lengths_dict.items():
        if len(lengths) > 0:
            np.save(os.path.join(DRIVE_RESULTS_DIR, f'lkh3_lengths_tsp_{n}.npy'), lengths)
            
    config_dict = {
        'timestamp': time.time(),
        'tsp_sizes': TSP_SIZES,
        'num_eval_instances': NUM_EVAL_INSTANCES,
        'lkh_runs': LKH_RUNS,
        'lkh_max_trials': LKH_MAX_TRIALS,
        'time_limit_sec': TIME_LIMIT_SEC,
        'seed': SEED
    }
    with open(os.path.join(DRIVE_RESULTS_DIR, 'config_lkh3.json'), 'w') as f:
        json.dump(config_dict, f, indent=4)
        
    print("[OK] Results and config saved successfully.")

In [ ]:
if len(df_results) > 0:
    status_counts = df_results.groupby(['n', 'status']).size().unstack(fill_value=0)
    print("Solver Status Report:")
    print("-" * 60)
    print(status_counts)
    print("-" * 60)

In [ ]:
if len(df_results) > 0:
    # Pick a middle size from evaluated sizes for visualization
    eval_sizes = summary['n'].tolist()
    if eval_sizes:
        viz_n = eval_sizes[len(eval_sizes) // 2]
        
        val_path = os.path.join(DRIVE_DATA_DIR, f'tsp_{viz_n}_val.npy')
        if os.path.exists(val_path):
            dataset = np.load(val_path)
            sample_coords = dataset[0]
            
            res = solve_tsp_lkh(sample_coords, time_limit_sec=10, instance_id=1000)
            
            if res['status'] == 'SUCCESS':
                tour = res['tour']
                tour_coords = sample_coords[tour]
                tour_coords = np.vstack((tour_coords, tour_coords[0]))
                
                plt.figure(figsize=(6, 6))
                plt.plot(tour_coords[:, 0], tour_coords[:, 1], marker='o', linestyle='-', color='b')
                plt.plot(tour_coords[0, 0], tour_coords[0, 1], marker='s', color='r', markersize=10, label='Start/End')
                plt.title(f'LKH-3 Tour for TSP-{viz_n} (Length: {res["tour_length"]:.4f})')
                plt.legend()
                plt.grid(True)
                plt.show()
            else:
                print("[WARNING] Could not solve instance for visualization.")

## Output Structure
Running this notebook generates the following artifacts in `DRIVE_RESULTS_DIR`:
- `lkh3_per_instance.csv`: Detailed results for every run.
- `lkh3_summary.csv`: Aggregated metrics (mean length, mean time, success rate) per problem size.
- `lkh3_lengths_tsp_{n}.npy`: Arrays of valid tour lengths for downstream statistical analysis.
- `config_lkh3.json`: The hyperparameter config used for the run.

### Loading Example
```python
import pandas as pd
import numpy as np

# Load summary
summary_df = pd.read_csv('/content/drive/MyDrive/TSP_Data/baseline_results/lkh3_summary.csv')

# Load specific tour lengths
lengths_50 = np.load('/content/drive/MyDrive/TSP_Data/baseline_results/lkh3_lengths_tsp_50.npy')
```